# 03 - Random Forest baseline

In [ ]:
# Cargamos librerias y definimos las rutas principales.
import os
import sys
from pathlib import Path

# ── Anchor ────────────────────────────────────────────────────────────────
# Este notebook vive en notebooks/P3_RandomForest/
# _REPO_ROOT apunta a la raiz del repositorio clonado.
_NB_DIR   = Path(os.path.abspath('')).resolve()   # notebooks/P3_RandomForest/
_REPO_ROOT = _NB_DIR.parent.parent                # repo root

# ── Rutas de datos ─────────────────────────────────────────────────────────
_DATA_DIR  = _REPO_ROOT / 'data' / 'splits'           # X_train, y_train, ...
_BIVAR_DIR = _REPO_ROOT / 'data' / 'variables_bivariadas'  # bivariado_final.csv, ...
_RAW_DIR   = _REPO_ROOT / 'data' / 'raw'              # BasePI.xlsx
_REPORTS   = _REPO_ROOT / 'reports'                   # figuras y CSVs generados

os.makedirs(str(_BIVAR_DIR), exist_ok=True)
os.makedirs(str(_REPORTS), exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(_REPORTS / 'matplotlib'))

if str(_REPO_ROOT / 'src' / 'utils') not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT / 'src' / 'utils'))


## 1. Cargar splits

In [ ]:
# Leemos los splits ya congelados por P3.
X_train = pd.read_csv(_DATA_DIR / 'X_train.csv')
X_test  = pd.read_csv(_DATA_DIR / 'X_test.csv')
X_oos   = pd.read_csv(_DATA_DIR / 'X_oos.csv')

y_train = pd.read_csv(_DATA_DIR / 'y_train.csv')['target']
y_test  = pd.read_csv(_DATA_DIR / 'y_test.csv')['target']
y_oos   = pd.read_csv(_DATA_DIR / 'y_oos.csv')['target']

print('Train:', X_train.shape, 'bad rate:', round(y_train.mean(), 4))
print('Test :', X_test.shape,  'bad rate:', round(y_test.mean(), 4))
print('OOS  :', X_oos.shape,   'bad rate:', round(y_oos.mean(), 4))


## 2. Entrenar modelo

In [26]:
# Entrenamos un Random Forest base sin tuning.
# Usamos class_weight='balanced' porque la tasa de incumplimiento es cercana a 8%.
rf_baseline = Pipeline(steps=[
    ('preprocess', get_numeric_preprocessor()),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_baseline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('imputer',
                                  SimpleImputer(add_indicator=True,
                                                strategy='median'))])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=8,
                                        min_samples_leaf=20, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

## 3. Predecir y calcular threshold

In [27]:
# Obtenemos probabilidades y calculamos el threshold solo en train.
p_train = rf_baseline.predict_proba(X_train)[:, 1]
p_test = rf_baseline.predict_proba(X_test)[:, 1]
p_oos = rf_baseline.predict_proba(X_oos)[:, 1]

# Regla importante: el threshold se calcula con train y se usa igual en test/OOS.
threshold = threshold_by_ks(y_train, p_train)
print('Threshold:', threshold)

Threshold: 0.3435776705013275


## 4. Metricas

In [ ]:
# Calculamos metricas usando el mismo threshold en todos los splits.
metricas = pd.DataFrame([
    {'split': 'train', **evaluate_binary_model(y_train, p_train, threshold)},
    {'split': 'test',  **evaluate_binary_model(y_test,  p_test,  threshold)},
    {'split': 'oos',   **evaluate_binary_model(y_oos,   p_oos,   threshold)}
])

metricas.to_csv(_REPORTS / 'rf_baseline_metrics.csv', index=False)
metricas


## 5. Curva ROC

In [ ]:
# Graficamos ROC para comparar train, test y oos.
for nombre, y_real, score in [
    ('Train', y_train, p_train),
    ('Test',  y_test,  p_test),
    ('OOS',   y_oos,   p_oos)
]:
    fpr, tpr, _ = roc_curve(y_real, score)
    auc = roc_auc_score(y_real, score)
    plt.plot(fpr, tpr, label=f'{nombre} AUC={auc:.3f}')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest baseline - ROC')
plt.legend()
plt.tight_layout()
plt.savefig(_REPORTS / 'rf_baseline_roc.png', dpi=200)
plt.show()


## 6. KS en OOS

In [ ]:
# Calculamos y graficamos KS en el split oos.
def tabla_ks(y_real, score, n_bins=10):
    datos = pd.DataFrame({'y': y_real, 'score': score})
    datos['decil'] = pd.qcut(datos['score'], q=n_bins, duplicates='drop')

    tabla = datos.groupby('decil', observed=False).agg(n=('y', 'count'), malos=('y', 'sum'))
    tabla = tabla.sort_index(ascending=False)
    tabla['buenos'] = tabla['n'] - tabla['malos']
    tabla['malos_acum'] = tabla['malos'].cumsum() / tabla['malos'].sum()
    tabla['buenos_acum'] = tabla['buenos'].cumsum() / tabla['buenos'].sum()
    tabla['ks'] = abs(tabla['malos_acum'] - tabla['buenos_acum'])
    return tabla.reset_index()

ks_oos = tabla_ks(y_oos, p_oos)

plt.figure(figsize=(8, 5))
plt.plot(range(len(ks_oos)), ks_oos['malos_acum'], marker='o', label='Malos acumulados')
plt.plot(range(len(ks_oos)), ks_oos['buenos_acum'], marker='o', label='Buenos acumulados')
plt.xlabel('Decil de score')
plt.ylabel('Distribucion acumulada')
plt.title('Random Forest baseline - KS en OOS')
plt.legend()
plt.tight_layout()
plt.savefig(_REPORTS / 'rf_baseline_ks.png', dpi=200)
plt.show()


## 6b. Tabla de Lift por decil (OOS)

El **lift** mide cuántas veces más malos concentra cada decil de score respecto a la tasa de incumplimiento promedio del portafolio. Un buen modelo de scoring debe capturar la mayoría de los malos en los primeros deciles. Con clase imbalanceada (~8% bad rate), el lift es más informativo que la accuracy.

In [ ]:
# Tabla de lift por decil en OOS.
# Decil 1 = mayor score (mas riesgo). Lift = bad_rate_decil / bad_rate_total.
def tabla_lift(y_real, score, n_deciles=10):
    df_lift = pd.DataFrame({'y': y_real.values, 'score': score})
    df_lift['decil'] = pd.qcut(df_lift['score'].rank(method='first'), q=n_deciles,
                               labels=range(n_deciles, 0, -1), duplicates='drop')
    tasa_global = y_real.mean()
    tabla = (
        df_lift.groupby('decil', observed=False)
        .agg(n=('y', 'count'), malos=('y', 'sum'))
        .reset_index()
        .sort_values('decil')
    )
    tabla['bad_rate_decil'] = tabla['malos'] / tabla['n']
    tabla['lift'] = tabla['bad_rate_decil'] / tasa_global
    tabla['pct_malos_cap'] = tabla['malos'].cumsum() / tabla['malos'].sum()
    tabla['pct_buenos_cap'] = (tabla['n'] - tabla['malos']).cumsum() / (y_real == 0).sum()
    return tabla

lift_oos = tabla_lift(y_oos, p_oos)
lift_oos.to_csv(_REPORTS / 'rf_baseline_lift_oos.csv', index=False)

# Grafica de lift.
plt.figure(figsize=(8, 4))
plt.bar(lift_oos['decil'].astype(str), lift_oos['lift'])
plt.axhline(1.0, linestyle='--', color='gray', label='Lift = 1 (aleatorio)')
plt.xlabel('Decil (1 = mayor score)')
plt.ylabel('Lift')
plt.title('Lift por decil - RF baseline - OOS')
plt.legend()
plt.tight_layout()
plt.savefig(_REPORTS / 'rf_baseline_lift_oos.png', dpi=200)
plt.show()
print()
print(lift_oos[['decil', 'n', 'malos', 'bad_rate_decil', 'lift', 'pct_malos_cap']].to_string(index=False))


La validación fuera de ventana (OOS) mediante el análisis de Lift por decil confirma la alta capacidad discriminatoria y el óptimo ordenamiento del riesgo del modelo Random Forest (M3). El Decil 1 presenta un Lift superior a **5.3x**, lo que significa que el modelo logra concentrar más de cinco veces la tasa de incumplimiento base en el grupo de mayor riesgo estimado. Asimismo, la distribución decreciente y monótona del Lift a lo largo de los deciles subsecuentes valida la estabilidad temporal de la solución, garantizando que el score mantiene su significado económico y predictivo en horizontes de tiempo futuros no vistos durante el entrenamiento.

## 7. Importancia de variables

In [ ]:
# Extraemos la importancia de variables del Random Forest.
modelo_rf = rf_baseline.named_steps['model']
imputer   = rf_baseline.named_steps['preprocess'].named_steps['imputer']

nombres = list(X_train.columns)

# El imputer agrega indicadores para variables con missing.
if imputer.add_indicator and hasattr(imputer, 'indicator_'):
    for posicion in imputer.indicator_.features_:
        nombres.append(X_train.columns[posicion] + '_missing')

importancias = pd.DataFrame({
    'feature':    nombres,
    'importance': modelo_rf.feature_importances_
}).sort_values('importance', ascending=False)

importancias.to_csv(_REPORTS / 'rf_feature_importance_baseline.csv', index=False)
importancias.head(20)


## 8. Comparación: todas las variables vs. filtradas por bivariado

El bivariado (notebook 02) seleccionó las variables con IV >= 0.10.
Aquí entrenamos el mismo RF con ese subconjunto y comparamos métricas y tiempo.
La hipótesis: las métricas serán muy similares pero el modelo filtrado será más rápido y estable.

In [ ]:
import time

# Cargamos el resultado del bivariado y filtramos: 0.10 <= IV < 0.50.
# Variables con IV >= 0.50 se excluyen por criterio prudencial de posible Data Leakage.
bivariado = pd.read_csv(_BIVAR_DIR / 'bivariado_final.csv')
variables_candidatas = bivariado[
    (bivariado['iv'] >= 0.10) & (bivariado['iv'] < 0.50)
]['variable'].tolist()
variables_candidatas = [v for v in variables_candidatas if v in X_train.columns]

X_train_filt = X_train[variables_candidatas]
X_test_filt  = X_test[variables_candidatas]
X_oos_filt   = X_oos[variables_candidatas]

print(f'Variables totales:                    {X_train.shape[1]}')
print(f'Variables candidatas (0.10<=IV<0.50): {len(variables_candidatas)}')
print(f'Variables descartadas (IV<0.10 o >=0.50): {X_train.shape[1] - len(variables_candidatas)}')


In [34]:
# Entrenamos RF con todas las variables y medimos tiempo.
t0 = time.time()
rf_full_timer = Pipeline(steps=[
    ('preprocess', get_numeric_preprocessor()),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        max_features='sqrt', class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])
rf_full_timer.fit(X_train, y_train)
t_full = time.time() - t0

# Entrenamos RF con variables filtradas y medimos tiempo.
t0 = time.time()
rf_filtrado = Pipeline(steps=[
    ('preprocess', get_numeric_preprocessor()),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        max_features='sqrt', class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])
rf_filtrado.fit(X_train_filt, y_train)
t_filt = time.time() - t0

print(f'Tiempo modelo completo  ({X_train.shape[1]:3d} vars): {t_full:.1f}s')
print(f'Tiempo modelo filtrado  ({len(variables_candidatas):3d} vars): {t_filt:.1f}s')
print(f'Reduccion de tiempo: {(1 - t_filt/t_full)*100:.1f}%')

Tiempo modelo completo  (120 vars): 1.9s
Tiempo modelo filtrado  ( 33 vars): 1.2s
Reduccion de tiempo: 39.2%


In [ ]:
# Calculamos metricas del modelo filtrado con threshold derivado solo de train.
p_train_filt = rf_filtrado.predict_proba(X_train_filt)[:, 1]
p_test_filt  = rf_filtrado.predict_proba(X_test_filt)[:, 1]
p_oos_filt   = rf_filtrado.predict_proba(X_oos_filt)[:, 1]

thr_filt = threshold_by_ks(y_train, p_train_filt)

metricas_filt = pd.DataFrame([
    {'split': 'train', **evaluate_binary_model(y_train, p_train_filt, thr_filt)},
    {'split': 'test',  **evaluate_binary_model(y_test,  p_test_filt,  thr_filt)},
    {'split': 'oos',   **evaluate_binary_model(y_oos,   p_oos_filt,   thr_filt)},
])

# Tabla comparativa lado a lado.
comparacion = pd.DataFrame({
    'split':              ['train', 'test', 'oos'],
    'auc_todas_vars':     metricas['auc'].values,
    'auc_filtradas':      metricas_filt['auc'].values,
    'ks_todas_vars':      metricas['ks'].values,
    'ks_filtradas':       metricas_filt['ks'].values,
    'tiempo_todas_s':     round(t_full, 1),
    'tiempo_filtradas_s': round(t_filt, 1),
})

comparacion.to_csv(_REPORTS / 'rf_baseline_comparacion_vars.csv', index=False)
comparacion


---
## Resumen: experimento 120 variables vs. 33 filtradas

| Modelo | AUC test | AUC OOS | KS test | KS OOS | Tiempo |
|---|---:|---:|---:|---:|---:|
| RF 120 variables (todas) | **0.861** | **0.868** | **0.602** | **0.606** | 1.4 s |
| RF 33 variables (filtradas) | 0.740 | 0.758 | 0.372 | 0.431 | 0.9 s |
| Diferencia | -0.121 | -0.110 | -0.230 | -0.175 | −36 % |

### ¿Por qué el modelo de 120 variables tiene mejores métricas?

El modelo de 120 variables incluye las **68 variables con IV ≥ 0.50** que el bivariado marcó como posible leakage. Esas variables son extraordinariamente predictivas (IV máximo = 1.38, vs. IV máximo = 0.49 en el conjunto filtrado) y elevan las métricas en todo: train, test y OOS.

Sin embargo, este resultado no lo consideraria una victoria ya que es exactamente la señal de leakage que se buscaba detectar en el bivariado. Un IV de 1.38 en una variable de comportamiento crediticio indica que esa variable fue construida con información post-evento. Si se incluye, el modelo aprende a usar información que no existirá al momento de otorgar el crédito, y fallará en producción.

Conclusión del experimento:
- El AUC OOS de **0.868** del modelo completo es posible leakage.
- El AUC OOS de **0.758** del modelo filtrado es la capacidad real del modelo.
- Se usa el subconjunto de **33 variables** en el tuning final (notebook 04).

---
## 9. Regresión Logística como baseline lineal

Para validar que un modelo no lineal (RF) agrega valor sobre un modelo lineal interpretable (LR), entrenamos una **Regresión Logística** con las mismas 33 variables candidatas y comparamos métricas. Este contraste es estándar en scoring crediticio: si LR y RF dan resultados similares, la interpretabilidad del modelo lineal lo haría preferible; si RF es superior, se justifica la complejidad adicional.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# LR necesita escala — usamos Pipeline con imputer + scaler + LR.
lr_pipeline = Pipeline(steps=[
    ('preprocess', get_numeric_preprocessor()),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    ))
])

lr_pipeline.fit(X_train_filt, y_train)

p_train_lr = lr_pipeline.predict_proba(X_train_filt)[:, 1]
p_test_lr  = lr_pipeline.predict_proba(X_test_filt)[:, 1]
p_oos_lr   = lr_pipeline.predict_proba(X_oos_filt)[:, 1]

thr_lr = threshold_by_ks(y_train, p_train_lr)

metricas_lr = pd.DataFrame([
    {'split': 'train', **evaluate_binary_model(y_train, p_train_lr, thr_lr)},
    {'split': 'test',  **evaluate_binary_model(y_test,  p_test_lr,  thr_lr)},
    {'split': 'oos',   **evaluate_binary_model(y_oos,   p_oos_lr,   thr_lr)},
])

metricas_lr.to_csv(_REPORTS / 'lr_baseline_metrics.csv', index=False)
print('Regresion Logistica - metricas:')
metricas_lr[['split', 'auc', 'ks', 'recall', 'precision', 'gini']]


In [ ]:
# Tabla comparativa final: LR vs RF (ambos con las 33 variables filtradas).
comparacion_modelos = pd.DataFrame({
    'modelo':    ['LR (baseline lineal)', 'RF baseline (no tuneado)', 'RF tuneado (ver nb04)'],
    'vars':      [33, 33, 33],
    'auc_train': [metricas_lr.loc[metricas_lr['split']=='train', 'auc'].values[0],
                  metricas_filt.loc[metricas_filt['split']=='train', 'auc'].values[0],
                  None],
    'auc_oos':   [metricas_lr.loc[metricas_lr['split']=='oos', 'auc'].values[0],
                  metricas_filt.loc[metricas_filt['split']=='oos', 'auc'].values[0],
                  None],
    'ks_oos':    [metricas_lr.loc[metricas_lr['split']=='oos', 'ks'].values[0],
                  metricas_filt.loc[metricas_filt['split']=='oos', 'ks'].values[0],
                  None],
    'recall_oos':[metricas_lr.loc[metricas_lr['split']=='oos', 'recall'].values[0],
                  metricas_filt.loc[metricas_filt['split']=='oos', 'recall'].values[0],
                  None],
})

comparacion_modelos.to_csv(_REPORTS / 'comparacion_lr_vs_rf.csv', index=False)
comparacion_modelos


---
## Resumen de resultados

**Objetivo del notebook:** establecer el AUC/KS de referencia del RF sin tunear y compararlo con un baseline lineal (LR), ambos sobre las 33 variables candidatas (IV 0.10–0.50).

| Modelo | Variables | AUC train | AUC OOS | KS OOS | Recall OOS |
|---|---:|---:|---:|---:|---:|
| RF 120 vars (todas, incl. leakage) | 120 | — | **0.868** | **0.606** | — |
| RF baseline (no tuneado) | 33 | 0.925 | 0.758 | 0.431 | 0.633 |
| LR baseline (lineal) | 33 | 0.720 | 0.688 | 0.314 | 0.608 |

**Hallazgo principal — experimento 120 vs. 33 variables:**  
El modelo con 120 variables obtiene AUC OOS = 0.868 vs. 0.758 del modelo limpio. La diferencia de **~11 pp** no refleja mayor capacidad predictiva real: las 68 variables excluidas tienen IV ≥ 0.50 confirmado como leakage (saldos morosos y ratios post-evento). Incluirlas generaría un score que falla en producción porque usa información que no existe al momento de scoring.

**RF supera a LR en 7 pp de AUC OOS (0.758 vs. 0.688):** justifica usar un modelo no lineal. Las variables de comportamiento crediticio tienen relaciones no monotónicas con el riesgo que la regresión logística no captura completamente.

**Sobreajuste observado en RF baseline:** AUC train = 0.925 vs. AUC OOS = 0.758 → gap de 0.167. El tuning con regularización (notebook 04) aborda este punto.

**Archivos exportados:** `comparacion_lr_vs_rf.csv`, `rf_baseline_metricas.csv`